In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a radix sort algorithm that sorts an array of 32-bit unsigned integers on a GPU.
    The program should take an input array of unsigned integers and sort them in ascending order using the radix sort algorithm.
    The <code>input</code> parameter contains the unsorted array, and the sorted result should be stored in the <code>output</code> array.
  </p>

  <h2>Implementation Requirements</h2>
  <ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The final sorted result must be stored in the <code>output</code> array</li>
    <li>Use radix sort algorithm (not other sorting algorithms)</li>
    <li>Sort in ascending order</li>
  </ul>

  <h2>Example 1:</h2>
  <pre>
  Input:  [170, 45, 75, 90, 2, 802, 24, 66]
  Output: [2, 24, 45, 66, 75, 90, 170, 802]
  </pre>

  <h2>Example 2:</h2>
  <pre>
  Input:  [1, 4, 1, 3, 555, 1000, 2]
  Output: [1, 1, 2, 3, 4, 555, 1000]
  </pre>

  <h2>Constraints</h2>
  <ul>
    <li><code>1 ≤ N ≤ 100,000,000</code></li>
    <li><code>0 ≤ input[i] ≤ 4,294,967,295</code> (32-bit unsigned integers)</li>

  <li>Performance is measured with <code>N</code> = 50,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers
extern "C" void solve(const unsigned int* input, unsigned int* output, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input is a tensor on the GPU
@jax.jit
def solve(input: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[UInt32, MutExternalOrigin],
    output: UnsafePointer[UInt32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def radix_sort_kernel(input, output, N):
    input = input.to(tl.pointer_type(tl.uint32))
    output = output.to(tl.pointer_type(tl.uint32))


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(name="Radix Sort", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free")

    def reference_impl(self, input: torch.Tensor, output: torch.Tensor, N: int):

        assert input.dtype == torch.uint32
        assert output.dtype == torch.uint32
        assert input.shape == output.shape == (N,)

        # Convert uint32 to int64 for sorting (since torch.sort doesn't support uint32)
        input_int64 = input.to(torch.int64)
        sorted_tensor = torch.sort(input_int64)[0]
        # Convert back to uint32
        output.copy_(sorted_tensor.to(torch.uint32))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_uint32), "in"),
            "output": (ctypes.POINTER(ctypes.c_uint32), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.uint32
        N = 8
        input_data = torch.tensor([170, 45, 75, 90, 2, 802, 24, 66], device="cuda", dtype=dtype)
        output_data = torch.zeros(N, device="cuda", dtype=dtype)
        return {
            "input": input_data,
            "output": output_data,
            "N": N,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.uint32
        test_cases = []

        # Test case 1: basic example
        test_cases.append(
            {
                "input": torch.tensor(
                    [170, 45, 75, 90, 2, 802, 24, 66], device="cuda", dtype=dtype
                ),
                "output": torch.zeros(8, device="cuda", dtype=dtype),
                "N": 8,
            }
        )

        # Test case 2: duplicate numbers
        test_cases.append(
            {
                "input": torch.tensor([1, 4, 1, 3, 555, 1000, 2], device="cuda", dtype=dtype),
                "output": torch.zeros(7, device="cuda", dtype=dtype),
                "N": 7,
            }
        )

        # Test case 3: single element
        test_cases.append(
            {
                "input": torch.tensor([42], device="cuda", dtype=dtype),
                "output": torch.zeros(1, device="cuda", dtype=dtype),
                "N": 1,
            }
        )

        # Test case 4: already sorted
        test_cases.append(
            {
                "input": torch.tensor([1, 2, 3, 4, 5, 6], device="cuda", dtype=dtype),
                "output": torch.zeros(6, device="cuda", dtype=dtype),
                "N": 6,
            }
        )

        # Test case 5: reverse sorted
        test_cases.append(
            {
                "input": torch.tensor([6, 5, 4, 3, 2, 1], device="cuda", dtype=dtype),
                "output": torch.zeros(6, device="cuda", dtype=dtype),
                "N": 6,
            }
        )

        # Test case 6: large numbers
        test_cases.append(
            {
                "input": torch.tensor(
                    [4294967295, 1000000000, 500000000, 2000000000, 100000000],
                    device="cuda",
                    dtype=dtype,
                ),
                "output": torch.zeros(5, device="cuda", dtype=dtype),
                "N": 5,
            }
        )

        # Test case 7: medium random
        test_cases.append(
            {
                "input": torch.randint(0, 1000001, (1024,), device="cuda", dtype=dtype),
                "output": torch.zeros(1024, device="cuda", dtype=dtype),
                "N": 1024,
            }
        )

        # Test case 8: large random
        test_cases.append(
            {
                "input": torch.randint(0, 4294967296, (10000,), device="cuda", dtype=dtype),
                "output": torch.zeros(10000, device="cuda", dtype=dtype),
                "N": 10000,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.uint32
        N = 50000000
        return {
            "input": torch.randint(0, 4294967296, (N,), device="cuda", dtype=dtype),
            "output": torch.zeros(N, device="cuda", dtype=dtype),
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
